# Notebook 9: Final Summary & Comprehensive Comparison
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

This notebook aggregates results from all experiments and generates comprehensive comparison visualizations for the research paper.

**Experiments Covered:**
1. Same-stock prediction (80/20 and 70/30)
2. Cross-stock prediction (80/20 and 70/30)
3. Different timeframe training (80/20 and 70/30)
4. Multi-stock training (80/20 and 70/30)

**Key Comparisons:**
- BiLSTM vs BiGRU (main comparison)
- BiLSTM/BiGRU vs LSTM/GRU (bidirectional vs unidirectional)
- 80/20 vs 70/30 split performance
- Cross-stock generalization ability
- Timeframe transfer learning


In [29]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

set_ieee_style()

os.makedirs('figures/summary', exist_ok=True)
os.makedirs('results', exist_ok=True)

print("Setup complete!")


Setup complete!


## 1. Load All Experiment Results

In [30]:
# ============================================================
# LOAD ALL RESULTS CSVs
# ============================================================
result_files = {
    'Exp1_80_20': 'results/Exp1_80_20_results.csv',
    'Exp1_70_30': 'results/Exp1_70_30_results.csv',
    'Exp2_80_20': 'results/Exp2_80_20_results.csv',
    'Exp2_70_30': 'results/Exp2_70_30_results.csv',
    'Exp3_80_20': 'results/Exp3_80_20_results.csv',
    'Exp3_70_30': 'results/Exp3_70_30_results.csv',
    'Exp4_80_20': 'results/Exp4_80_20_results.csv',
    'Exp4_70_30': 'results/Exp4_70_30_results.csv',
}

results = {}
for key, filepath in result_files.items():
    if os.path.exists(filepath):
        results[key] = pd.read_csv(filepath)
        print(f"  Loaded: {key} ({len(results[key])} rows)")
    else:
        print(f"  WARNING: {filepath} not found. Run the corresponding notebook first.")

print(f"\nLoaded {len(results)} result files.")


  Loaded: Exp1_80_20 (16 rows)
  Loaded: Exp1_70_30 (16 rows)
  Loaded: Exp2_80_20 (48 rows)
  Loaded: Exp2_70_30 (48 rows)
  Loaded: Exp3_80_20 (48 rows)
  Loaded: Exp3_70_30 (48 rows)
  Loaded: Exp4_80_20 (16 rows)
  Loaded: Exp4_70_30 (16 rows)

Loaded 8 result files.


In [ ]:
# ============================================================
# FRIEDMAN TEST - Non-parametric comparison of models
# ============================================================
# The Friedman test compares multiple models (BiLSTM, BiGRU, LSTM, GRU)
# across groups (stocks, targets) to detect significant performance differences

print("\n" + "=" * 80)
print("  FRIEDMAN TEST: Statistical Comparison of Models")
print("=" * 80)

# Store Friedman results for all experiments
friedman_summary = []

for exp_name, df in sorted(results.items()):
    print(f"\n{'─'*80}")
    print(f"  {exp_name}")
    print(f"{'─'*80}")
    
    # Determine grouping column based on experiment type
    if 'Stock' in df.columns and 'Target_Stock' not in df.columns:
        group_col = 'Stock'  # Exp1, Exp3
    elif 'Target_Stock' in df.columns:
        group_col = 'Target_Stock'  # Exp4
    elif 'Train_Stock' in df.columns:
        group_col = 'Test_Stock'  # Exp2
    else:
        print(f"  Skipping - cannot determine grouping column")
        continue
    
    # Run Friedman tests for all metrics
    metrics_to_test = []
    for metric in ['RMSE', 'MAE', 'R2']:
        if metric in df.columns:
            metrics_to_test.append(metric)
    if 'MAPE (%)' in df.columns:
        metrics_to_test.append('MAPE (%)')
    
    for metric in metrics_to_test:
        try:
            friedman_result = perform_friedman_test(df, metric=metric, group_by=group_col)
            
            # Print results
            sig_str = "✓ SIGNIFICANT" if friedman_result['Significant (α=0.05)'] == 'Yes' else "✗ NOT SIGNIFICANT"
            print(f"\n  {metric} (grouped by {group_col}):")
            print(f"    Test Statistic: {friedman_result['Test Statistic']:.6f}")
            print(f"    p-value: {friedman_result['p-value']}")
            print(f"    Result: {sig_str}")
            
            # Store for summary
            friedman_summary.append({
                'Experiment': exp_name,
                'Metric': metric,
                'Group_By': group_col,
                'Test_Statistic': friedman_result['Test Statistic'],
                'p_value': float(friedman_result['p-value']),
                'Significant': friedman_result['Significant (α=0.05)'],
            })
        except Exception as e:
            print(f"    Error: {str(e)}")

# Create summary DataFrame
if friedman_summary:
    friedman_df = pd.DataFrame(friedman_summary)
    
    print(f"\n{'='*80}")
    print("  FRIEDMAN TEST SUMMARY")
    print(f"{'='*80}")
    print(friedman_df.to_string(index=False))
    
    # Save to CSV
    friedman_df.to_csv('results/friedman_test_summary.csv', index=False)
    print(f"\nSaved to: results/friedman_test_summary.csv")


## 1.5 Statistical Testing - Friedman Test

## 2. Experiment 1: Same-Stock Prediction — 80/20 vs 70/30

In [31]:
# ============================================================
# COMPARE 80/20 vs 70/30 for Same-Stock Prediction
# ============================================================
if 'Exp1_80_20' in results and 'Exp1_70_30' in results:
    df_80 = results['Exp1_80_20'].copy()
    df_80['Split'] = '80/20'
    df_70 = results['Exp1_70_30'].copy()
    df_70['Split'] = '70/30'
    
    combined = pd.concat([df_80, df_70], ignore_index=True)
    
    # Print comparison
    print("=" * 80)
    print("  EXPERIMENT 1: Same-Stock Prediction - 80/20 vs 70/30")
    print("=" * 80)
    
    for stock in STOCKS:
        print(f"\n--- {stock} ---")
        stock_data = combined[combined['Stock'] == stock]
        pivot = stock_data.pivot_table(
            values=['RMSE', 'MAE', 'MAPE (%)', 'R2'],
            index='Model', columns='Split'
        )
        print(pivot.round(4).to_string())
    
    # Grouped bar chart: RMSE comparison
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Experiment 1: Same-Stock Prediction\nRMSE — 80/20 vs 70/30', 
                 fontsize=20, fontweight='bold')
    
    for idx, stock in enumerate(STOCKS):
        ax = axes[idx // 2, idx % 2]
        stock_data = combined[combined['Stock'] == stock]
        
        x = np.arange(len(MODEL_TYPES))
        w = 0.35
        
        vals_80 = [stock_data[(stock_data['Model'] == m) & (stock_data['Split'] == '80/20')]['RMSE'].values
                   for m in MODEL_TYPES]
        vals_70 = [stock_data[(stock_data['Model'] == m) & (stock_data['Split'] == '70/30')]['RMSE'].values
                   for m in MODEL_TYPES]
        
        vals_80 = [v[0] if len(v) > 0 else 0 for v in vals_80]
        vals_70 = [v[0] if len(v) > 0 else 0 for v in vals_70]
        
        bars1 = ax.bar(x - w/2, vals_80, w, label='80/20', color='#0072B2', alpha=0.8)
        bars2 = ax.bar(x + w/2, vals_70, w, label='70/30', color='#D55E00', alpha=0.8)
        
        ax.set_title(f'{stock}', fontsize=20)
        ax.set_xticks(x)
        ax.set_xticklabels(MODEL_TYPES, fontsize=20)
        ax.set_ylabel('RMSE', fontsize=20)
        ax.legend(fontsize=20)
        
        # Value labels
        for bar in bars1:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                    f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=20)
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                    f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=20)
    
    fig.tight_layout()
    save_fig(fig, 'figures/summary/exp1_rmse_80vs70.png')
    
    # R² comparison
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Experiment 1: Same-Stock Prediction\nR² Score — 80/20 vs 70/30',
                 fontsize=20, fontweight='bold')
    
    for idx, stock in enumerate(STOCKS):
        ax = axes[idx // 2, idx % 2]
        stock_data = combined[combined['Stock'] == stock]
        
        x = np.arange(len(MODEL_TYPES))
        w = 0.35
        
        vals_80 = [stock_data[(stock_data['Model'] == m) & (stock_data['Split'] == '80/20')]['R2'].values
                   for m in MODEL_TYPES]
        vals_70 = [stock_data[(stock_data['Model'] == m) & (stock_data['Split'] == '70/30')]['R2'].values
                   for m in MODEL_TYPES]
        
        vals_80 = [v[0] if len(v) > 0 else 0 for v in vals_80]
        vals_70 = [v[0] if len(v) > 0 else 0 for v in vals_70]
        
        bars1 = ax.bar(x - w/2, vals_80, w, label='80/20', color='#0072B2', alpha=0.8)
        bars2 = ax.bar(x + w/2, vals_70, w, label='70/30', color='#D55E00', alpha=0.8)
        
        ax.set_title(f'{stock}', fontsize=20)
        ax.set_xticks(x)
        ax.set_xticklabels(MODEL_TYPES, fontsize=20)
        ax.set_ylabel('R² Score', fontsize=20)
        ax.legend(fontsize=20)
    
    fig.tight_layout()
    save_fig(fig, 'figures/summary/exp1_r2_80vs70.png')
    
    print("\nExp1 comparison plots saved!")
else:
    print("Missing Exp1 results. Run Notebooks 1 and 5 first.")


  EXPERIMENT 1: Same-Stock Prediction - 80/20 vs 70/30

--- TLKM ---
            MAE          MAPE (%)              R2             RMSE          
Split     70/30    80/20    70/30   80/20   70/30   80/20    70/30     80/20
Model                                                                       
BiGRU   42.7629  61.7345   1.4986  1.9777  0.9849  0.9662  56.3155   76.0037
BiLSTM  40.7985  46.1028   1.4128  1.4871  0.9857  0.9786  54.8735   60.5278
GRU     41.0255  98.4776   1.4177  3.1107  0.9856  0.9279  55.1425  110.9943
LSTM    43.6870  45.0634   1.4931  1.4501  0.9840  0.9794  58.0689   59.3935

--- BBCA ---
             MAE           MAPE (%)              R2              RMSE          
Split      70/30     80/20    70/30   80/20   70/30   80/20     70/30     80/20
Model                                                                          
BiGRU   105.0418   95.9070   1.5128  1.1553  0.9931  0.9844  131.4567  129.9185
BiLSTM  136.6641   92.3103   1.7859  1.1336  0.9875  0.986

## 3. BiLSTM vs BiGRU — Main Comparison

In [32]:
# ============================================================
# MAIN COMPARISON: BiLSTM vs BiGRU across ALL experiments
# ============================================================
if 'Exp1_80_20' in results:
    # Filter to only BiLSTM and BiGRU
    bilstm_bigru = results['Exp1_80_20'][
        results['Exp1_80_20']['Model'].isin(['BiLSTM', 'BiGRU'])
    ].copy()
    
    print("=" * 60)
    print("  BiLSTM vs BiGRU — Same-Stock Prediction (80/20)")
    print("=" * 60)
    
    pivot = bilstm_bigru.pivot_table(
        values=['MSE', 'RMSE', 'MAE', 'MAPE (%)', 'R2'],
        index='Stock', columns='Model'
    )
    print(pivot.round(4).to_string())
    
    # Radar/Spider chart for BiLSTM vs BiGRU
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    fig.suptitle('BiLSTM vs BiGRU — Per Stock Performance (80/20)',
                 fontsize=20, fontweight='bold')
    
    metrics_list = ['RMSE', 'MAE', 'MAPE (%)', 'R2']
    
    for idx, stock in enumerate(STOCKS):
        ax = axes[idx // 2, idx % 2]
        
        bilstm_data = bilstm_bigru[
            (bilstm_bigru['Stock'] == stock) & (bilstm_bigru['Model'] == 'BiLSTM')
        ]
        bigru_data = bilstm_bigru[
            (bilstm_bigru['Stock'] == stock) & (bilstm_bigru['Model'] == 'BiGRU')
        ]
        
        x = np.arange(len(metrics_list))
        w = 0.35
        
        bilstm_vals = [bilstm_data[m].values[0] if len(bilstm_data) > 0 else 0 for m in metrics_list]
        bigru_vals = [bigru_data[m].values[0] if len(bigru_data) > 0 else 0 for m in metrics_list]
        
        ax.bar(x - w/2, bilstm_vals, w, label='BiLSTM', color=MODEL_COLORS['BiLSTM'])
        ax.bar(x + w/2, bigru_vals, w, label='BiGRU', color=MODEL_COLORS['BiGRU'])
        
        ax.set_title(f'{stock}', fontsize=20)
        ax.set_xticks(x)
        ax.set_xticklabels(metrics_list, fontsize=20)
        ax.legend(fontsize=20)
    
    fig.tight_layout()
    save_fig(fig, 'figures/summary/bilstm_vs_bigru_same_stock.png')
    print("BiLSTM vs BiGRU comparison saved!")


  BiLSTM vs BiGRU — Same-Stock Prediction (80/20)
           MAE          MAPE (%)                 MSE                  R2              RMSE          
Model    BiGRU   BiLSTM    BiGRU  BiLSTM       BiGRU      BiLSTM   BiGRU  BiLSTM     BiGRU    BiLSTM
Stock                                                                                               
ASII   59.6656  62.8965   1.2691  1.3310   6663.1111   7344.9718  0.9820  0.9801   81.6279   85.7028
BBCA   95.9070  92.3103   1.1553  1.1336  16878.8110  15157.9388  0.9844  0.9860  129.9185  123.1176
TLKM   61.7345  46.1028   1.9777  1.4871   5776.5608   3663.6151  0.9662  0.9786   76.0037   60.5278
UNVR   49.0076  72.3127   1.8578  2.6237   5134.5209   8113.8431  0.9941  0.9907   71.6556   90.0769
  Figure saved: figures/summary/bilstm_vs_bigru_same_stock.png
BiLSTM vs BiGRU comparison saved!


## 4. Bidirectional vs Unidirectional Comparison

In [33]:
# ============================================================
# BIDIRECTIONAL vs UNIDIRECTIONAL
# ============================================================
if 'Exp1_80_20' in results:
    df = results['Exp1_80_20'].copy()
    df['Direction'] = df['Model'].apply(
        lambda x: 'Bidirectional' if x.startswith('Bi') else 'Unidirectional'
    )
    
    print("=" * 60)
    print("  Bidirectional vs Unidirectional — Average Across Stocks")
    print("=" * 60)
    
    dir_avg = df.groupby(['Direction', 'Model'])[['RMSE', 'MAE', 'MAPE (%)', 'R2']].mean()
    print(dir_avg.round(4).to_string())
    
    # Bar chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Bidirectional vs Unidirectional RNN — Average Performance',
                 fontsize=20, fontweight='bold')
    
    avg_by_model = df.groupby('Model')[['RMSE', 'R2']].mean()
    
    # RMSE
    ax = axes[0]
    models = MODEL_TYPES
    rmse_vals = [avg_by_model.loc[m, 'RMSE'] for m in models]
    bars = ax.bar(models, rmse_vals, color=[MODEL_COLORS[m] for m in models], 
                  edgecolor='white', linewidth=0.5)
    ax.set_ylabel('RMSE (avg)', fontsize=20)
    ax.set_title('Average RMSE', fontsize=20)
    for bar, val in zip(bars, rmse_vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f'{val:.2f}', ha='center', va='bottom', fontsize=20)
    
    # R²
    ax = axes[1]
    r2_vals = [avg_by_model.loc[m, 'R2'] for m in models]
    bars = ax.bar(models, r2_vals, color=[MODEL_COLORS[m] for m in models],
                  edgecolor='white', linewidth=0.5)
    ax.set_ylabel('R² Score (avg)', fontsize=20)
    ax.set_title('Average R² Score', fontsize=20)
    for bar, val in zip(bars, r2_vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f'{val:.4f}', ha='center', va='bottom', fontsize=20)
    
    fig.tight_layout()
    save_fig(fig, 'figures/summary/bidirectional_vs_unidirectional.png')
    print("Direction comparison saved!")


  Bidirectional vs Unidirectional — Average Across Stocks
                           RMSE       MAE  MAPE (%)      R2
Direction      Model                                       
Bidirectional  BiGRU    89.8014   66.5787    1.5650  0.9817
               BiLSTM   89.8563   68.4056    1.6438  0.9838
Unidirectional GRU     126.0965  104.1508    2.4361  0.9636
               LSTM    104.8156   80.4258    1.6902  0.9790
  Figure saved: figures/summary/bidirectional_vs_unidirectional.png
Direction comparison saved!


## 5. Cross-Stock Generalization Analysis

In [34]:
# ============================================================
# CROSS-STOCK GENERALIZATION ANALYSIS
# ============================================================
if 'Exp2_80_20' in results:
    df = results['Exp2_80_20'].copy()
    
    print("=" * 60)
    print("  Cross-Stock Generalization — Average RMSE per Model")
    print("=" * 60)
    avg_cross = df.groupby('Model')[['RMSE', 'MAE', 'MAPE (%)', 'R2']].mean()
    print(avg_cross.round(4).to_string())
    
    # Compare same-stock vs cross-stock performance
    if 'Exp1_80_20' in results:
        same_avg = results['Exp1_80_20'].groupby('Model')['RMSE'].mean()
        cross_avg = df.groupby('Model')['RMSE'].mean()
        
        fig, ax = plt.subplots(figsize=(10, 6))
        x = np.arange(len(MODEL_TYPES))
        w = 0.35
        
        same_vals = [same_avg[m] for m in MODEL_TYPES]
        cross_vals = [cross_avg[m] for m in MODEL_TYPES]
        
        ax.bar(x - w/2, same_vals, w, label='Same-Stock (Exp1)', color='#0072B2', alpha=0.8)
        ax.bar(x + w/2, cross_vals, w, label='Cross-Stock (Exp2)', color='#D55E00', alpha=0.8)
        
        ax.set_xticks(x)
        ax.set_xticklabels(MODEL_TYPES, fontsize=20)
        ax.set_ylabel('Average RMSE', fontsize=20)
        ax.set_title('Same-Stock vs Cross-Stock Prediction\nAverage RMSE (80/20)',
                     fontsize=20)
        ax.legend(fontsize=20)
        fig.tight_layout()
        save_fig(fig, 'figures/summary/same_vs_cross_stock_rmse.png')
        print("Same vs Cross-stock comparison saved!")


  Cross-Stock Generalization — Average RMSE per Model
            RMSE      MAE  MAPE (%)      R2
Model                                      
BiGRU   106.7298  84.0830    1.7246  0.9766
BiLSTM  102.6521  79.7641    1.6535  0.9786
GRU     104.2314  81.6440    1.6932  0.9771
LSTM    111.3741  88.0283    1.7397  0.9743
  Figure saved: figures/summary/same_vs_cross_stock_rmse.png
Same vs Cross-stock comparison saved!


## 6. Multi-Stock Training Analysis

In [35]:
# ============================================================
# MULTI-STOCK TRAINING ANALYSIS
# ============================================================
if 'Exp4_80_20' in results:
    df = results['Exp4_80_20'].copy()
    
    print("=" * 60)
    print("  Multi-Stock Training — Results")
    print("=" * 60)
    print(df.to_string(index=False))
    
    # Compare Exp1 (same-stock) vs Exp4 (multi-stock)
    if 'Exp1_80_20' in results:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle('Same-Stock (Exp1) vs Multi-Stock Training (Exp4)\nRMSE Comparison (80/20)',
                     fontsize=20, fontweight='bold')
        
        for idx, target in enumerate(STOCKS):
            ax = axes[idx // 2, idx % 2]
            
            # Same-stock RMSE
            same = results['Exp1_80_20'][results['Exp1_80_20']['Stock'] == target]
            # Multi-stock RMSE
            multi = df[df['Target_Stock'] == target]
            
            x = np.arange(len(MODEL_TYPES))
            w = 0.35
            
            s_vals = [same[same['Model'] == m]['RMSE'].values[0] if len(same[same['Model'] == m]) > 0 else 0 
                      for m in MODEL_TYPES]
            m_vals = [multi[multi['Model'] == m]['RMSE'].values[0] if len(multi[multi['Model'] == m]) > 0 else 0 
                      for m in MODEL_TYPES]
            
            ax.bar(x - w/2, s_vals, w, label='Same-Stock', color='#0072B2', alpha=0.8)
            ax.bar(x + w/2, m_vals, w, label='Multi-Stock', color='#D55E00', alpha=0.8)
            
            ax.set_title(f'{target}', fontsize=20)
            ax.set_xticks(x)
            ax.set_xticklabels(MODEL_TYPES, fontsize=20)
            ax.set_ylabel('RMSE', fontsize=20)
            ax.legend(fontsize=20)
        
        fig.tight_layout()
        save_fig(fig, 'figures/summary/same_vs_multi_stock_rmse.png')
        print("Same vs Multi-stock comparison saved!")


  Multi-Stock Training — Results
  Train_Stocks Target_Stock  Model         MSE     RMSE      MAE  MAPE (%)       R2  Training_Time_s  Epochs_Run
BBCA+ASII+UNVR         TLKM BiLSTM   3074.2899  55.4463  41.0255    1.3320 0.982004            246.5         100
BBCA+ASII+UNVR         TLKM  BiGRU   3207.6424  56.6361  42.3410    1.3677 0.981224            209.4         100
BBCA+ASII+UNVR         TLKM   LSTM   4139.2717  64.3372  49.5355    1.6109 0.975771            153.0         100
BBCA+ASII+UNVR         TLKM    GRU   3478.5592  58.9793  44.5900    1.4384 0.979638            132.4         100
TLKM+ASII+UNVR         BBCA BiLSTM  61825.8183 248.6480 197.1576    2.2711 0.942799            261.7         100
TLKM+ASII+UNVR         BBCA  BiGRU  38244.5027 195.5620 153.1168    1.7853 0.964616            230.1         100
TLKM+ASII+UNVR         BBCA   LSTM 114586.3467 338.5061 256.8757    2.9074 0.893984            144.8         100
TLKM+ASII+UNVR         BBCA    GRU  54515.7332 233.4860 178.754

## 7. Comprehensive Results Table

In [36]:
# ============================================================
# MASTER RESULTS TABLE
# ============================================================
print("\n" + "=" * 80)
print("  COMPREHENSIVE RESULTS SUMMARY")
print("=" * 80)

for exp_name, df in sorted(results.items()):
    print(f"\n{'─'*60}")
    print(f"  {exp_name}")
    print(f"{'─'*60}")
    
    # Best model
    if 'RMSE' in df.columns:
        valid = df.dropna(subset=['RMSE'])
        if not valid.empty:
            best_idx = valid['RMSE'].idxmin()
            best = valid.loc[best_idx]
            print(f"  Best by RMSE: {best['Model']} (RMSE={best['RMSE']:.4f})")
            
            worst_idx = valid['RMSE'].idxmax()
            worst = valid.loc[worst_idx]
            print(f"  Worst by RMSE: {worst['Model']} (RMSE={worst['RMSE']:.4f})")
    
    # Average per model
    if 'Model' in df.columns:
        avg = df.groupby('Model')[['RMSE', 'MAE', 'MAPE (%)', 'R2']].mean()
        print(f"\n  Average metrics per model:")
        print(avg.round(4).to_string())



  COMPREHENSIVE RESULTS SUMMARY

────────────────────────────────────────────────────────────
  Exp1_70_30
────────────────────────────────────────────────────────────
  Best by RMSE: BiLSTM (RMSE=54.8735)
  Worst by RMSE: LSTM (RMSE=219.7358)

  Average metrics per model:
            RMSE      MAE  MAPE (%)      R2
Model                                      
BiGRU   104.4460  82.4496    1.8896  0.9860
BiLSTM  102.4844  75.9152    1.5908  0.9893
GRU     100.2525  73.4178    1.5729  0.9890
LSTM    114.2622  86.2414    1.7414  0.9866

────────────────────────────────────────────────────────────
  Exp1_80_20
────────────────────────────────────────────────────────────
  Best by RMSE: LSTM (RMSE=59.3935)
  Worst by RMSE: GRU (RMSE=213.7833)

  Average metrics per model:
            RMSE       MAE  MAPE (%)      R2
Model                                       
BiGRU    89.8014   66.5787    1.5650  0.9817
BiLSTM   89.8563   68.4056    1.6438  0.9838
GRU     126.0965  104.1508    2.4361  0.96

In [ ]:
# ============================================================
# VISUALIZE FRIEDMAN TEST RESULTS
# ============================================================
print("\nGenerating Friedman test visualizations...")

# Create visualizations for each experiment
for exp_name, df in sorted(results.items()):
    print(f"  Processing {exp_name}...", end=" ")
    
    # Determine grouping column
    if 'Stock' in df.columns and 'Target_Stock' not in df.columns:
        group_col = 'Stock'
    elif 'Target_Stock' in df.columns:
        group_col = 'Target_Stock'
    elif 'Train_Stock' in df.columns:
        group_col = 'Test_Stock'
    else:
        print("skipped")
        continue
    
    try:
        # Run Friedman tests for all metrics
        metrics_to_test = []
        for metric in ['RMSE', 'MAE', 'R2']:
            if metric in df.columns:
                metrics_to_test.append(metric)
        if 'MAPE (%)' in df.columns:
            metrics_to_test.append('MAPE (%)')
        
        friedman_results = []
        for metric in metrics_to_test:
            result = perform_friedman_test(df, metric=metric, group_by=group_col)
            friedman_results.append(result)
        
        # Create visualization
        fig, html_file = create_friedman_test_visualization(
            friedman_results,
            experiment_label=exp_name,
            save_dir='figures/summary'
        )
        print(f"✓ saved to {html_file}")
    except Exception as e:
        print(f"error: {str(e)}")

print("\nAll Friedman test visualizations complete!")


## 7.5 Friedman Test Visualizations

## 8. Overall Winner Analysis

In [37]:
# ============================================================
# OVERALL WINNER ANALYSIS
# ============================================================
print("\n" + "=" * 80)
print("  OVERALL MODEL RANKING")
print("=" * 80)

# Collect average RMSE and R² across all experiments
model_scores = {m: {'rmse_sum': 0, 'r2_sum': 0, 'count': 0} for m in MODEL_TYPES}

for exp_name, df in results.items():
    if 'Model' not in df.columns:
        continue
    for m in MODEL_TYPES:
        model_data = df[df['Model'] == m].dropna(subset=['RMSE', 'R2'])
        if not model_data.empty:
            model_scores[m]['rmse_sum'] += model_data['RMSE'].mean()
            model_scores[m]['r2_sum'] += model_data['R2'].mean()
            model_scores[m]['count'] += 1

# Compute overall averages
overall = []
for m in MODEL_TYPES:
    if model_scores[m]['count'] > 0:
        overall.append({
            'Model': m,
            'Avg_RMSE': model_scores[m]['rmse_sum'] / model_scores[m]['count'],
            'Avg_R2': model_scores[m]['r2_sum'] / model_scores[m]['count'],
            'N_Experiments': model_scores[m]['count'],
        })

overall_df = pd.DataFrame(overall)
overall_df = overall_df.sort_values('Avg_RMSE')
print(overall_df.to_string(index=False))

# Final ranking plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Overall Model Ranking Across All Experiments',
             fontsize=20, fontweight='bold')

# By RMSE (lower is better)
ax = axes[0]
sorted_df = overall_df.sort_values('Avg_RMSE')
bars = ax.barh(sorted_df['Model'], sorted_df['Avg_RMSE'],
               color=[MODEL_COLORS[m] for m in sorted_df['Model']])
ax.set_xlabel('Average RMSE (lower is better)', fontsize=20)
ax.set_title('Ranked by RMSE', fontsize=20)
for bar, val in zip(bars, sorted_df['Avg_RMSE']):
    ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2,
            f' {val:.2f}', va='center', fontsize=20)

# By R² (higher is better)
ax = axes[1]
sorted_df = overall_df.sort_values('Avg_R2', ascending=True)
bars = ax.barh(sorted_df['Model'], sorted_df['Avg_R2'],
               color=[MODEL_COLORS[m] for m in sorted_df['Model']])
ax.set_xlabel('Average R² (higher is better)', fontsize=20)
ax.set_title('Ranked by R²', fontsize=20)
for bar, val in zip(bars, sorted_df['Avg_R2']):
    ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2,
            f' {val:.4f}', va='center', fontsize=20)

fig.tight_layout()
save_fig(fig, 'figures/summary/overall_model_ranking.png')

print("\n\nOverall ranking plot saved!")
print("\n" + "=" * 40)
print("  WINNER: " + overall_df.iloc[0]['Model'])
print("=" * 40)



  OVERALL MODEL RANKING
 Model   Avg_RMSE   Avg_R2  N_Experiments
 BiGRU 153.022201 0.909759              8
   GRU 158.024326 0.914580              8
BiLSTM 172.413291 0.876882              8
  LSTM 212.181939 0.797480              8
  Figure saved: figures/summary/overall_model_ranking.png


Overall ranking plot saved!

  WINNER: BiGRU


## 9. Conclusion

All results and figures have been saved:

**Results CSVs:** `results/` directory  
**Figures:** `figures/summary/` directory (600 DPI, PNG format)

**Key Figure Files:**
- `exp1_rmse_80vs70.png` — 80/20 vs 70/30 RMSE comparison
- `exp1_r2_80vs70.png` — 80/20 vs 70/30 R² comparison
- `bilstm_vs_bigru_same_stock.png` — Main model comparison
- `bidirectional_vs_unidirectional.png` — Bi- vs Uni-directional
- `same_vs_cross_stock_rmse.png` — Generalization analysis
- `same_vs_multi_stock_rmse.png` — Multi-stock training effect
- `overall_model_ranking.png` — Final ranking

These figures are formatted for IEEE publication at 600 DPI.


In [ ]:
# ============================================================
# FRIEDMAN TEST INTERPRETATION
# ============================================================
print("\n" + "=" * 80)
print("  FRIEDMAN TEST INTERPRETATION & RECOMMENDATIONS")
print("=" * 80)

if 'friedman_df' in locals() and not friedman_df.empty:
    # Separate significant and non-significant results
    significant = friedman_df[friedman_df['Significant'] == 'Yes']
    non_significant = friedman_df[friedman_df['Significant'] == 'No']
    
    print(f"\nTOTAL TESTS PERFORMED: {len(friedman_df)}")
    print(f"  Significant Results (p < 0.05): {len(significant)} tests")
    print(f"  Non-Significant Results (p ≥ 0.05): {len(non_significant)} tests")
    
    if len(significant) > 0:
        print(f"\n{'─'*80}")
        print("  SIGNIFICANT DIFFERENCES DETECTED:")
        print(f"{'─'*80}")
        for _, row in significant.iterrows():
            print(f"\n  {row['Experiment']} - {row['Metric']} (grouped by {row['Group_By']})")
            print(f"    Test Statistic: {row['Test_Statistic']:.6f}")
            print(f"    p-value: {row['p_value']:.6f}")
            print(f"    ➜ Models perform DIFFERENTLY on this metric/group combination")
            print(f"    ➜ Recommendation: Use post-hoc tests (e.g., Nemenyi) to identify pairwise differences")
    
    if len(non_significant) > 0:
        print(f"\n{'─'*80}")
        print("  NO SIGNIFICANT DIFFERENCES DETECTED:")
        print(f"{'─'*80}")
        for _, row in non_significant.iterrows():
            print(f"\n  {row['Experiment']} - {row['Metric']} (grouped by {row['Group_By']})")
            print(f"    Test Statistic: {row['Test_Statistic']:.6f}")
            print(f"    p-value: {row['p_value']:.6f}")
            print(f"    ➜ No significant difference among models on this metric")
            print(f"    ➜ Recommendation: Models perform equivalently; choose based on computational cost")
    
    # Summary statistics by experiment
    print(f"\n{'─'*80}")
    print("  SUMMARY BY EXPERIMENT:")
    print(f"{'─'*80}")
    
    for exp in friedman_df['Experiment'].unique():
        exp_data = friedman_df[friedman_df['Experiment'] == exp]
        sig_count = len(exp_data[exp_data['Significant'] == 'Yes'])
        total_count = len(exp_data)
        
        print(f"\n  {exp}:")
        print(f"    Significant tests: {sig_count}/{total_count}")
        
        if sig_count > 0:
            sig_metrics = exp_data[exp_data['Significant'] == 'Yes']['Metric'].unique()
            print(f"    Metrics with differences: {', '.join(sig_metrics)}")
else:
    print("\nNo Friedman test results available. Run the statistical testing cell first.")

print(f"\n{'='*80}")
print("  NOTE: Friedman test checks if models differ significantly.")
print("  - p < 0.05: Models perform DIFFERENTLY (at least one is different)")
print("  - p ≥ 0.05: No significant difference (models perform similarly)")
print(f"{'='*80}\n")


## 8. Friedman Test Interpretation & Recommendations